# Stop Guessing, Start Measuring
## Evaluating RAG with synthetic test data

This notebook mirrors the **current** RAGAS docs, as of 19.06.2026. 
It does four things:

1. Builds a deliberately *simple* RAG system over the public **Hugging Face docs** corpus.
2. **Generates a synthetic test set** from those docs with RAGAS's **knowledge-graph** pipeline.
3. Runs the RAG over the test set.
4. **Evaluates** it with the five RAG metrics via the current `ragas.metrics.collections` API:
   **Context Precision, Context Recall, Faithfulness, Answer Relevancy, Noise Sensitivity.**


## 0. Setup

In [ ]:
import os, getpass
from dotenv import load_dotenv

load_dotenv()
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

import ragas
from ragas.testset import TestsetGenerator
from ragas.metrics.collections import (
    Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall, NoiseSensitivity,
)
from ragas.llms import llm_factory
from ragas.embeddings import OpenAIEmbeddings as RagasOpenAIEmbeddings
print("ragas", ragas.__version__, "- imports OK")


ragas 0.4.3 - imports OK


## 1. A tiny RAG over the Hugging Face docs

Deliberately minimal — OpenAI embeddings for retrieval, OpenAI chat for generation. The talk is
about *evaluating* the RAG, not building a fancy one. Swap in your own pipeline here later.

In [ ]:
from datasets import load_dataset
from langchain_core.documents import Document

# Hugging Face docs corpus
ds = load_dataset("m-ric/huggingface_doc", split="train")
subset = ds.select(range(10))  # keep it small for the demo
documents = [Document(page_content=r["text"], metadata={"source": r.get("source", "")})
             for r in subset]
print(len(documents), "documents")


10 documents


### Peek at the source documents (Hugging Face docs)

Browse the corpus in the dataset viewer: **https://huggingface.co/datasets/m-ric/huggingface_doc**
— a scrape of the official docs at https://huggingface.co/docs . Each row is one page (`text`)
plus its `source`.

In [27]:
print("columns:", subset.column_names)
for r in subset.select(range(2)):
    print("\nsource:", r.get("source"))
    print(r["text"][:300], "...")


columns: ['text', 'source']

source: huggingface/hf-endpoints-documentation/blob/main/docs/source/guides/create_endpoint.mdx
 Create an Endpoint

After your first login, you will be directed to the [Endpoint creation page](https://ui.endpoints.huggingface.co/new). As an example, this guide will go through the steps to deploy [distilbert-base-uncased-finetuned-sst-2-english](https://huggingface.co/distilbert-base-uncased-f ...

source: huggingface/evaluate/blob/main/docs/source/choosing_a_metric.mdx
 Choosing a metric for your task

**So you've trained your model and want to see how well it’s doing on a dataset of your choice. Where do you start?**

There is no “one size fits all” approach to choosing an evaluation metric, but some good guidelines to keep in mind are:

## Categories of metrics
 ...


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import numpy as np
from openai import OpenAI

client = OpenAI()
EMBED_MODEL = "text-embedding-3-small"
GEN_MODEL = "gpt-4o-mini"

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = [c.page_content for c in splitter.split_documents(documents)]
print(len(chunks), "chunks")

def embed(texts):
    resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return np.array([d.embedding for d in resp.data], dtype=np.float32)

# Embed the corpus once (this is the only bulk embedding call).
chunk_emb = embed(chunks)
chunk_emb /= (np.linalg.norm(chunk_emb, axis=1, keepdims=True) + 1e-9)


59 chunks


In [30]:
def retrieve(query, k=4):
    q = embed([query])[0]
    q /= (np.linalg.norm(q) + 1e-9)
    sims = chunk_emb @ q
    return [chunks[i] for i in sims.argsort()[::-1][:k]]

def rag_answer(query, k=4):
    ctx = retrieve(query, k)
    context = "\n\n".join(f"[Doc {i+1}] {c}" for i, c in enumerate(ctx))
    msg = [
        {"role": "system", "content": "Answer ONLY from the provided documents. Be concise. "
                                       "If the answer is not in the documents, say so."},
        {"role": "user", "content": f"Documents:\n{context}\n\nQuestion: {query}"},
    ]
    out = client.chat.completions.create(model=GEN_MODEL, messages=msg)
    return out.choices[0].message.content, ctx

ans, ctx = rag_answer("What is a tokenizer, in a single sentence?")
print(ans)


A tokenizer is a tool that converts text into a format suitable for model training, typically by breaking it down into smaller units like words or subwords.


## 2. Generate a synthetic test set with RAGAS (knowledge graph)

RAGAS chunks the docs into a **knowledge graph** (nodes = chunks, edges = shared entities/relationships), then traverses it to manufacture diverse questions —
including **multi-hop** ones that join two pages, the edge cases a human would never hand-write.

Each generated row gives us **`user_input`**, **`reference`** (gold answer), and **`reference_contexts`** — the labelled test item we otherwise couldn't build by hand.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.testset import TestsetGenerator
from ragas.testset.graph import KnowledgeGraph, Node, NodeType
from ragas.testset.transforms import apply_transforms, default_transforms_for_prechunked

generator = TestsetGenerator.from_langchain(
    ChatOpenAI(model="gpt-4o-mini"),
    OpenAIEmbeddings(model="text-embedding-3-small"),
)

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
doc_chunks = splitter.split_documents(documents)

kg = KnowledgeGraph()
for d in doc_chunks:
    kg.nodes.append(
        Node(
            type=NodeType.CHUNK,
            properties={"page_content": d.page_content, "document_metadata": d.metadata},
        )
    )

# Build the KG: extract entities/themes/summaries -> relationships
transforms = default_transforms_for_prechunked(generator.llm, generator.embedding_model)
apply_transforms(kg, transforms)
generator.knowledge_graph = kg

# Traverse the KG to synthesize queries (single-hop + multi-hop).
testset = generator.generate(testset_size=8)

df = testset.to_pandas()
df[["user_input", "reference"]]


Applying SummaryExtractor:   0%|          | 0/59 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/59 [00:00<?, ?it/s]

Node bda302c8-e71b-4cfc-9d1a-50b932fbcfce does not have a summary. Skipping filtering.
Node 1e49c0e1-3bcd-490a-aca4-7d39d995653e does not have a summary. Skipping filtering.
Node 86201863-3bcb-40bd-b198-b761c0567edd does not have a summary. Skipping filtering.
Node 3e5e4298-8d41-44dc-b87a-37dfd12d2a38 does not have a summary. Skipping filtering.
Node 019ad3a2-2fdd-4858-933d-974057325bac does not have a summary. Skipping filtering.
Node 4e708e66-06db-41be-9273-6dad6055c08b does not have a summary. Skipping filtering.
Node 35fff4af-3454-405f-a6f5-34581836519a does not have a summary. Skipping filtering.
Node de3785c0-9b02-4129-a318-0ad5b5452f94 does not have a summary. Skipping filtering.
Node 7b1e31b6-87ea-4863-93b1-2d843464de70 does not have a summary. Skipping filtering.
Node e6978a77-ede2-4aa6-a4c3-c14ab4578c2f does not have a summary. Skipping filtering.
Node 151808c8-8efc-4f60-95ec-730ee0508f3c does not have a summary. Skipping filtering.
Node 925e2c04-87ed-43f0-947f-ba8f194bd259 d

Applying EmbeddingExtractor:   0%|          | 0/59 [00:00<?, ?it/s]

Applying ThemesExtractor:   0%|          | 0/59 [00:00<?, ?it/s]

...

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/9 [00:00<?, ?it/s]

,user_input,reference
0,How do you create an endpoint using Hugging Face?,"To create an endpoint using Hugging Face, firs..."
1,Can you explain the significance of the GLUE b...,The GLUE benchmark is significant as it provid...
2,How is accuracy used in evaluating datasets?,Accuracy is a metric that can be used for eval...
3,What is SE-ResNet and how does it relate to Sq...,SE-ResNet is a variant of ResNet that incorpor...
4,What steps are involved in managing SSH key au...,To manage SSH key authentication for Git opera...
5,What are the steps involved in model training ...,To fine-tune LayoutLMv3 for token classificati...
6,What is SE ResNet and what are its results for...,SE ResNet is a collection of models including ...
7,What are TPU Nodes and how do they affect data...,TPU Nodes are a way to access remote TPUs indi...
8,What role does ImageNet play in the training t...,ImageNet serves as the training data for image...


### Peek at what came out — including a multi-hop question

In [44]:
import pandas as pd
pd.set_option("display.max_colwidth", 140)

# 'synthesizer_name' shows which query type produced each row (single-hop vs multi-hop).
cols = [c for c in ["user_input", "reference", "synthesizer_name"] if c in df.columns]
display(df[cols])

# Show one full row (question + gold answer + reference contexts).
row0 = df.iloc[-1]
print("Question:", row0["user_input"])
print("\nReference answer:", row0["reference"])
print("\n# reference_contexts:", len(row0["reference_contexts"]))


,user_input,reference,synthesizer_name
0,How do you create an endpoint using Hugging Face?,"To create an endpoint using Hugging Face, first log in and go to the Endpoint creation page. Enter the Hugging Face Repository ID and yo...",single_hop_specific_query_synthesizer
1,Can you explain the significance of the GLUE benchmark in evaluating machine learning models?,The GLUE benchmark is significant as it provides a dedicated evaluation metric specifically designed to measure model performance on var...,single_hop_specific_query_synthesizer
2,How is accuracy used in evaluating datasets?,"Accuracy is a metric that can be used for evaluating labeled (supervised) datasets, providing a measure of how often the predictions mad...",single_hop_specific_query_synthesizer
3,What is SE-ResNet and how does it relate to Squeeze-and-Excitation Networks?,"SE-ResNet is a variant of ResNet that incorporates squeeze-and-excitation blocks, allowing the network to perform dynamic channel-wise f...",multi_hop_abstract_query_synthesizer
4,"What steps are involved in managing SSH key authentication for Git operations on huggingface.co, and how can you test the SSH connection...","To manage SSH key authentication for Git operations on huggingface.co, you first need to check for existing SSH keys on your local machi...",multi_hop_abstract_query_synthesizer
5,What are the steps involved in model training and how does the model upload process work when fine-tuning LayoutLMv3 on CORD?,"To fine-tune LayoutLMv3 for token classification on CORD, you would execute a series of commands in a Python script. The command include...",multi_hop_abstract_query_synthesizer
6,What is SE ResNet and what are its results for image classification?,"SE ResNet is a collection of models including seresnet50, which is designed for image classification tasks. The results for image classi...",multi_hop_specific_query_synthesizer
7,What are TPU Nodes and how do they affect data access when using them for Python code?,"TPU Nodes are a way to access remote TPUs indirectly, requiring a separate VM to initialize the network and data pipeline. When using TP...",multi_hop_specific_query_synthesizer
8,"What role does ImageNet play in the training techniques and resources for image classification models, specifically in relation to the u...","ImageNet serves as the training data for image classification models, which utilize various training techniques such as label smoothing,...",multi_hop_specific_query_synthesizer


Question: What role does ImageNet play in the training techniques and resources for image classification models, specifically in relation to the use of 8x NVIDIA Titan X GPUs?

Reference answer: ImageNet serves as the training data for image classification models, which utilize various training techniques such as label smoothing, SGD with momentum, and weight decay. The training resources specified include 8x NVIDIA Titan X GPUs, which are essential for handling the computational demands of training models like seresnet50 on the ImageNet dataset.

# reference_contexts: 2


### Inspect & save the knowledge graph

In [33]:
# The generator exposes the KG it built. Attribute name can vary slightly by version,
# so we guard it.
try:
    kg = generator.knowledge_graph
    print("nodes:", len(kg.nodes), "| relationships:", len(kg.relationships))
    sample_node = kg.nodes[0]
    print("example node properties:", list(sample_node.properties.keys()))
    kg.save("hf_docs_kg.json")   # persist -> render this for the 'show them the KG' slide
    print("saved -> hf_docs_kg.json")
except AttributeError as e:
    print("KG attribute differs in this ragas version:", e)
    print("See docs.ragas.io/en/stable/concepts/test_data_generation/rag/")


nodes: 59 | relationships: 96
example node properties: ['page_content', 'document_metadata', 'summary', 'summary_embedding', 'themes', 'entities']
saved -> hf_docs_kg.json


### Visualize the knowledge graph

Nodes are document chunks; edges connect chunks RAGAS found **related** — sharing named
entities/themes, or being semantically similar (cosine similarity of their summaries). Those edges
are exactly what let it build **multi-hop** questions that span two chunks.

In [37]:
# %pip install -q pyvis
from pyvis.network import Network
from IPython.display import IFrame

net = Network(height="600px", width="100%", bgcolor="#ffffff", font_color="#222",
              notebook=True, cdn_resources="in_line")
# Node label = the same number as the matplotlib legend; hover (title) shows the full chunk.
for n, d in G.nodes(data=True):
    net.add_node(n, label=str(d.get("idx", "")), title=d.get("snippet", ""),
                 color="#6c8cff", size=16)
# Edge label = relationship type, shown directly on the edge (hover too).
for a, b, d in G.edges(data=True):
    rt = d.get("rtype", "")
    net.add_edge(a, b, label=rt, title=rt, color="#94a3b8", font={"size": 10})
net.force_atlas_2based(gravity=-40, spring_length=130)
net.show("hf_docs_kg.html")
IFrame("hf_docs_kg.html", width="100%", height=620)

hf_docs_kg.html


### How to make a unique test set based on your requirements

Personas + a weighted query distribution let you steer the mix (e.g. 40% entity single-hop /
40% theme single-hop / 20% multi-hop).

In [38]:
from ragas.testset.persona import Persona
from ragas.testset.synthesizers.single_hop.specific import SingleHopSpecificQuerySynthesizer
from ragas.testset.synthesizers.multi_hop.specific import MultiHopSpecificQuerySynthesizer

persona = Persona(
    name="HF developer",
    role_description="A developer integrating a Hugging Face library, asking practical how-to "
                     "and API questions.",
)

query_distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=generator.llm, property_name="entities"), 0.4),
    (SingleHopSpecificQuerySynthesizer(llm=generator.llm, property_name="themes"),   0.4),
    (MultiHopSpecificQuerySynthesizer(llm=generator.llm),                            0.2),
]

# The KG was already built in the generation cell above, so we re-run generate() (not
# generate_with_langchain_docs) to apply the persona + distribution without rebuilding it.
# generator.persona_list = [persona]
# testset = generator.generate(testset_size=8, query_distribution=query_distribution)
print("personas + distribution configured (re-run generation to apply)")


personas + distribution configured (re-run generation to apply)


-------------------------
## 3. Run the RAG over the generated test set

In [39]:
records = []
for _, r in df.iterrows():
    response, retrieved = rag_answer(r["user_input"], k=4)
    records.append({
        "user_input": r["user_input"],
        "reference": r["reference"],
        "response": response,
        "retrieved_contexts": retrieved,
    })

eval_df = pd.DataFrame(records)
eval_df[["user_input", "response"]]


,user_input,response
0,How do you create an endpoint using Hugging Face?,"To create an endpoint using Hugging Face, follow these steps:\n\n1. Log in and go to the [Endpoint creation page](https://ui.endpoints.h..."
1,Can you explain the significance of the GLUE benchmark in evaluating machine learning models?,The GLUE benchmark is significant in evaluating machine learning models because it is a collection of different subsets focused on vario...
2,How is accuracy used in evaluating datasets?,Accuracy is used for evaluating labeled (supervised) datasets.
3,What is SE-ResNet and how does it relate to Squeeze-and-Excitation Networks?,"SE-ResNet is a variant of ResNet that incorporates squeeze-and-excitation blocks to perform dynamic channel-wise feature recalibration, ..."
4,"What steps are involved in managing SSH key authentication for Git operations on huggingface.co, and how can you test the SSH connection...","To manage SSH key authentication for Git operations on huggingface.co, follow these steps:\n\n1. **Add your SSH key to your huggingface...."
5,What are the steps involved in model training and how does the model upload process work when fine-tuning LayoutLMv3 on CORD?,"To fine-tune LayoutLMv3 on CORD, you need to follow these steps:\n\n1. Run the script `run_funsd_cord.py` with the following command:\n ..."
6,What is SE ResNet and what are its results for image classification?,SE ResNet is a variant of ResNet that uses squeeze-and-excitation blocks for dynamic channel-wise feature recalibration. For image class...
7,What are TPU Nodes and how do they affect data access when using them for Python code?,"TPU Nodes allow indirect access to a remote TPU, requiring a separate VM to initialize the network and data pipeline, which then forward..."
8,"What role does ImageNet play in the training techniques and resources for image classification models, specifically in relation to the u...",ImageNet serves as the training data for the image classification models and is utilized in conjunction with 8x NVIDIA Titan X GPUs as t...


## 4. Score with the five RAGAS metrics

A separate **judge** LLM scores each row. Note how each metric reads a different slice of the
trajectory — that's what lets you localize a failure to *retrieval* vs *generation*.

| Metric | Reads | Good = | Catches |
|---|---|---|---|
| **Context Precision** | question, contexts, reference | high | retriever returning / ranking junk |
| **Context Recall** | question, contexts, reference | high | retriever missing needed chunks |
| **Faithfulness** | question, response, contexts | high | hallucination / unsupported claims |
| **Answer Relevancy** *(docs: Response Relevancy)* | question, response | high | evasive / off-topic answers |
| **Noise Sensitivity** | question, response, reference, contexts | **low** | junk context throwing it off |

*Naming:* the class is **`AnswerRelevancy`**; the docs display it as **Response Relevancy**. **`ContextPrecision`** here is the **reference-based** variant — it reads `reference` (not `response`). The sibling `ContextPrecisionWithoutReference` uses `response` instead, for systems with no gold answer.

In [ ]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings import OpenAIEmbeddings as RagasOpenAIEmbeddings
from ragas.metrics.collections import (
    Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall, NoiseSensitivity,
)

aclient = AsyncOpenAI()
judge     = llm_factory("gpt-4o-mini", client=aclient)                            # evaluator ("judge") LLM
judge_emb = RagasOpenAIEmbeddings(client=aclient, model="text-embedding-3-small")  # AnswerRelevancy needs embeddings

faithfulness      = Faithfulness(llm=judge)
answer_relevancy  = AnswerRelevancy(llm=judge, embeddings=judge_emb)
context_precision = ContextPrecision(llm=judge)
context_recall    = ContextRecall(llm=judge)
noise_sensitivity = NoiseSensitivity(llm=judge)


In [41]:
# Top-level await works inside Jupyter. Each metric is an LLM-as-judge call.
async def score_row(row):
    return {
        "context_precision": (await context_precision.ascore(
            user_input=row["user_input"], reference=row["reference"],
            retrieved_contexts=row["retrieved_contexts"])).value,
        "context_recall": (await context_recall.ascore(
            user_input=row["user_input"], reference=row["reference"],
            retrieved_contexts=row["retrieved_contexts"])).value,
        "faithfulness": (await faithfulness.ascore(
            user_input=row["user_input"], response=row["response"],
            retrieved_contexts=row["retrieved_contexts"])).value,
        "answer_relevancy": (await answer_relevancy.ascore(
            user_input=row["user_input"], response=row["response"])).value,
        "noise_sensitivity": (await noise_sensitivity.ascore(
            user_input=row["user_input"], response=row["response"],
            reference=row["reference"], retrieved_contexts=row["retrieved_contexts"])).value,
    }

scores = []
for r in eval_df.to_dict("records"):
    scores.append(await score_row(r))   # top-level await is supported in Jupyter

results = pd.concat([eval_df, pd.DataFrame(scores)], axis=1)
results.to_csv("eval_results.csv", index=False)
results[["user_input", "context_precision", "context_recall",
         "faithfulness", "answer_relevancy", "noise_sensitivity"]]


,user_input,context_precision,context_recall,faithfulness,answer_relevancy,noise_sensitivity
0,How do you create an endpoint using Hugging Face?,1.000000,1.00,1.00,0.966498,0.125000
1,Can you explain the significance of the GLUE benchmark in evaluating machine learning models?,1.000000,1.00,0.75,0.963312,0.666667
2,How is accuracy used in evaluating datasets?,0.750000,0.50,1.00,0.911412,0.000000
3,What is SE-ResNet and how does it relate to Squeeze-and-Excitation Networks?,1.000000,1.00,0.75,0.787971,0.000000
4,"What steps are involved in managing SSH key authentication for Git operations on huggingface.co, and how can you test the SSH connection...",1.000000,1.00,1.00,0.896419,0.384615
5,What are the steps involved in model training and how does the model upload process work when fine-tuning LayoutLMv3 on CORD?,0.916667,0.75,1.00,0.803315,0.428571
6,What is SE ResNet and what are its results for image classification?,0.250000,1.00,1.00,0.709449,1.000000
7,What are TPU Nodes and how do they affect data access when using them for Python code?,1.000000,1.00,1.00,0.752625,0.500000
8,"What role does ImageNet play in the training techniques and resources for image classification models, specifically in relation to the u...",1.000000,1.00,1.00,0.680743,0.000000


In [42]:
metric_cols = ["context_precision", "context_recall", "faithfulness",
               "answer_relevancy", "noise_sensitivity"]
results[metric_cols].mean().round(3)

context_precision    0.880
context_recall       0.917
faithfulness         0.944
answer_relevancy     0.830
noise_sensitivity    0.345
dtype: float64